In [1]:
import pandas as pd
from astroquery.simbad import Simbad
from openpyxl import load_workbook
from openpyxl.styles import PatternFill


# 1. File names

input_file="Exoplanet 6059-b_filled_from_pscomppars_Finale.xlsx"
output_file="Exoplanet_6059_with_SIMBAD_fill_Finale.xlsx"


# 2. Load workbook data

df=pd.read_excel(input_file)

# Column name cleanup
df=df.rename(columns={"StellarRadius [Solar Radius]": "Stellar Radius [Solar Radius]"})

# Clean Host Name for matching
df["Host Name"] = (
    df["Host Name"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 3. Configure SIMBAD

custom_simbad=Simbad()
custom_simbad.add_votable_fields("sp_type", "ids")   # use new name

# 4. Get host stars

hosts=df["Host Name"].dropna().unique().tolist()

# 5. Query SIMBAD in batch

try:
    simbad_results = custom_simbad.query_objects(hosts)
except Exception as e:
    print("SIMBAD batch query failed:", e)
    simbad_results=None

# 6. Convert SIMBAD results to DataFrame

if simbad_results is not None:
    print("SIMBAD columns:", simbad_results.colnames) 
    simbad_df=simbad_results.to_pandas()
else:
    simbad_df=pd.DataFrame()

# 7. Normalize returned columns

if not simbad_df.empty:
    simbad_df.columns=[str(c).upper() for c in simbad_df.columns]

    # Make sure needed columns exist
    for col in ["MAIN_ID", "SP_TYPE", "IDS"]:
        if col not in simbad_df.columns:
            simbad_df[col]=pd.NA

    # Clean values
    for col in ["MAIN_ID", "SP_TYPE", "IDS"]:
        simbad_df[col]=simbad_df[col].astype("string")
        simbad_df[col]=simbad_df[col].replace(
            {
                "<NA>": pd.NA,
                "nan": pd.NA,
                "None": pd.NA,
                "MaskedConstant": pd.NA
            }
        )

    print("\nSample SIMBAD rows:")
    print(simbad_df[["MAIN_ID", "SP_TYPE", "IDS"]].head())

    print("\nNon-null SP_TYPE count in returned SIMBAD table:")
    print(simbad_df["SP_TYPE"].notna().sum())
else:
    simbad_df=pd.DataFrame(columns=["MAIN_ID", "SP_TYPE", "IDS"])

# 8. Helper functions

def normalize_name(name):
    if pd.isna(name):
        return ""
    return (
        str(name)
        .upper()
        .replace("-", " ")
        .replace("_", " ")
        .replace("  ", " ")
        .strip()
    )

def compact_name(name):
    return normalize_name(name).replace(" ", "")

def match_simbad_alias(host, simbad_df):
    host_clean=normalize_name(host)
    host_compact=compact_name(host)

    for _, row in simbad_df.iterrows():
        ids=row["IDS"]
        main_id=row["MAIN_ID"]
        sp_type=row["SP_TYPE"]

        ids_clean=normalize_name(ids) if pd.notna(ids) else ""
        ids_compact=ids_clean.replace(" ", "")

        main_id_clean=normalize_name(main_id) if pd.notna(main_id) else ""
        main_id_compact=main_id_clean.replace(" ", "")

        matched=(
            host_clean==main_id_clean
            or host_compact==main_id_compact
            or host_clean in ids_clean
            or host_compact in ids_compact
        )

        if matched:
            return sp_type, main_id

    return pd.NA, pd.NA

# 9. Match each host star to SIMBAD aliases

matched_rows=[]

for host in df["Host Name"]:
    sp_type, simbad_id=match_simbad_alias(host, simbad_df)
    matched_rows.append({
        "Host Name": host,
        "Star Spectral Type_SIMBAD": sp_type,
        "SIMBAD_ID": simbad_id
    })

simbad_final=pd.DataFrame(matched_rows)

unique_simbad_matches=(
    simbad_final[["Host Name", "Star Spectral Type_SIMBAD", "SIMBAD_ID"]]
    .drop_duplicates(subset=["Host Name"])
    .copy()
)

# 10. Merge row-aligned SIMBAD results

merged=df.copy()
merged["Star Spectral Type_SIMBAD"]=simbad_final["Star Spectral Type_SIMBAD"].values
merged["SIMBAD_ID"]=simbad_final["SIMBAD_ID"].values

# 11. Fill missing values from SIMBAD

columns_to_fill_from_simbad=[
    "Star Spectral Type"
]

fill_counts_simbad={}
source_tracker=pd.DataFrame(index=merged.index, columns=columns_to_fill_from_simbad)

for col in columns_to_fill_from_simbad:
    simbad_col=f"{col}_SIMBAD"

    before_missing=merged[col].isna().sum()

    fill_mask=merged[col].isna() & merged[simbad_col].notna()
    merged.loc[fill_mask, col]=merged.loc[fill_mask, simbad_col]
    source_tracker.loc[fill_mask, col]="SIMBAD"

    after_missing=merged[col].isna().sum()
    fill_counts_simbad[col]=before_missing - after_missing

filled_df=merged[df.columns].copy()

# 12. Save updated Excel file

filled_df.to_excel(output_file, index=False)

# 13. Highlight SIMBAD-filled cells

wb=load_workbook(output_file)
ws=wb.active

blue_fill=PatternFill(fill_type="solid", fgColor="BDD7EE")

header_map={cell.value: cell.column for cell in ws[1]}

for row_idx in range(2, len(filled_df) + 2):
    df_idx = row_idx - 2

    for col in columns_to_fill_from_simbad:
        orig_val=df.iloc[df_idx][col]
        new_val=filled_df.iloc[df_idx][col]

        if pd.isna(orig_val) and pd.notna(new_val):
            source=source_tracker.iloc[df_idx][col]
            if source=="SIMBAD":
                excel_col=header_map[col]
                ws.cell(row=row_idx, column=excel_col).fill = blue_fill

# 14. Add summary sheet

summary_sheet_name="SIMBAD Fill Summary"
if summary_sheet_name in wb.sheetnames:
    del wb[summary_sheet_name]

summary=wb.create_sheet(summary_sheet_name)

summary["A1"]="Source"
summary["B1"]="SIMBAD"

summary["A3"]="Column"
summary["B3"]="Values Filled"

r=4
for col, count in fill_counts_simbad.items():
    summary[f"A{r}"]=col
    summary[f"B{r}"]=count
    r += 1

summary[f"A{r+1}"]="Matching Key"
summary[f"B{r+1}"]="Host Name"

summary[f"A{r+2}"]="Matching Method"
summary[f"B{r+2}"]="Alias/identifier match using SIMBAD IDS and MAIN_ID"

summary[f"A{r+3}"]="Color Key"
summary[f"B{r+3}"]="Blue = filled from SIMBAD"

summary[f"A{r+5}"]="Total Host Rows"
summary[f"B{r+5}"]=len(df)

summary[f"A{r+6}"]="Unique Host Stars Queried"
summary[f"B{r+6}"]=len(hosts)

summary[f"A{r+7}"]="SIMBAD Rows Returned"
summary[f"B{r+7}"]=len(simbad_df)

summary[f"A{r+8}"]="Unique Host Matches with SP_TYPE"
summary[f"B{r+8}"]=unique_simbad_matches["Star Spectral Type_SIMBAD"].notna().sum()

# Debug sheet
debug_sheet_name="SIMBAD Matches"
if debug_sheet_name in wb.sheetnames:
    del wb[debug_sheet_name]

debug=wb.create_sheet(debug_sheet_name)
debug_headers=["Host Name", "Star Spectral Type_SIMBAD", "SIMBAD_ID"]
for c, h in enumerate(debug_headers, start=1):
    debug.cell(row=1, column=c, value=h)

for i, row in enumerate(unique_simbad_matches.itertuples(index=False), start=2):
    debug.cell(row=i, column=1, value=row[0])
    debug.cell(row=i, column=2, value=None if pd.isna(row[1]) else str(row[1]))
    debug.cell(row=i, column=3, value=None if pd.isna(row[2]) else str(row[2]))

# 15. Save workbook

wb.save(output_file)

# 16. Print results

print("\nDone.")
print("Input file:", input_file)
print("Saved:", output_file)
print("Total host rows:", len(df))
print("Unique host stars queried:", len(hosts))
print("SIMBAD rows returned:", len(simbad_df))
print("Unique host matches with spectral type:",
      unique_simbad_matches["Star Spectral Type_SIMBAD"].notna().sum())
print("SIMBAD fill counts:")
print(fill_counts_simbad)

SIMBAD columns: ['main_id', 'ra', 'dec', 'coo_err_maj', 'coo_err_min', 'coo_err_angle', 'coo_wavelength', 'coo_bibcode', 'sp_type', 'ids', 'user_specified_id', 'object_number_id']

Sample SIMBAD rows:
                   MAIN_ID     SP_TYPE  \
0                *  24 Sex        K0IV   
1               [PS78] 191       M3.5V   
2  2MASS J04414489+2301513        M8.7   
3                *  47 UMa  G1-VFe-0.5   
4                *  51 Peg        G2IV   

                                                 IDS  
0  TIC 1713457|HIP 50887|Gaia DR3 383089708039505...  
1  RAVE J012250.9-243951|SIPS J0122-2439|Gaia DR3...  
2  Gaia DR3 146487556211644544|2MASS J04414489+23...  
3  TIC 21535479|GJ 407|HIP 53721|Gaia DR3 7772543...  
4  TIC 139298196|GJ 882|Gaia DR3 2835207319109249...  

Non-null SP_TYPE count in returned SIMBAD table:
4535

Done.
Input file: Exoplanet 6059-b_filled_from_pscomppars_Finale.xlsx
Saved: Exoplanet_6059_with_SIMBAD_fill_Finale.xlsx
Total host rows: 6059
Unique host stars

In [6]:
test_hosts=["HD 209458", "51 Peg", "WASP-12", "TOI-700"]
test_results=custom_simbad.query_objects(test_hosts)
print(test_results.colnames)
print(test_results)

['main_id', 'ra', 'dec', 'coo_err_maj', 'coo_err_min', 'coo_err_angle', 'coo_wavelength', 'coo_bibcode', 'sp_type', 'ids', 'user_specified_id', 'object_number_id']
 main_id         ra       ... user_specified_id object_number_id
                deg       ...                                   
--------- --------------- ... ----------------- ----------------
HD 209458 330.79488644388 ...         HD 209458                1
*  51 Peg 344.36658535524 ...         51 Peg                   2
  WASP-12  97.63665253848 ...         WASP-12                  3
  TOI-700  97.09678555474 ...         TOI-700                  4


In [7]:
sf=pd.read_excel("Exoplanet_6059_with_SIMBAD_fill_Finale.xlsx")
sf.isna().sum()

Planet Name                                    0
Host Name                                      0
Default Flag                                   0
Number of Stars                                0
Number of Planets                              0
Discovery Method                               0
Soltype                                        0
Controversial Flag                             0
Orbital Period [Days]                        320
Orbit Semi-Major Axis [au]                   313
Planet Radius [Earth Radius]                  44
Planet Mass or Mass*sin(i) [Earth Mass]       26
Equilibrium Temperature [K]                 1504
Star Spectral Type                          2203
Stellar Effective Temperature [K]            281
Stellar Radius [Solar Radius]                299
Stellar Mass [Solar mass]                      5
Stellar Surface Gravity [log10(cm/s**2)]     305
Distance [pc]                                 27
dtype: int64

In [8]:
df.isna().sum()

Planet Name                                    0
Host Name                                      0
Default Flag                                   0
Number of Stars                                0
Number of Planets                              0
Discovery Method                               0
Soltype                                        0
Controversial Flag                             0
Orbital Period [Days]                        320
Orbit Semi-Major Axis [au]                   313
Planet Radius [Earth Radius]                  44
Planet Mass or Mass*sin(i) [Earth Mass]       26
Equilibrium Temperature [K]                 1504
Star Spectral Type                          3815
Stellar Effective Temperature [K]            281
Stellar Radius [Solar Radius]                299
Stellar Mass [Solar mass]                      5
Stellar Surface Gravity [log10(cm/s**2)]     305
Distance [pc]                                 27
dtype: int64

In [9]:
print("SIMBAD spectral types filled:", df["Star Spectral Type"].isna().sum() - sf["Star Spectral Type"].isna().sum())

SIMBAD spectral types filled: 1612
